# NB-03 | STRIDE Threat Analysis
**Pipeline Step 3 of 5**

Applies STRIDE methodology to every grounded component using Claude Sonnet. Each threat includes structured `iae_factors` (the nine IAE scoring dimensions as integers 1–3) so that NB04 can score from structured data rather than keyword-matching prose.

**Key improvements over v1:**
- LLM outputs `iae_factors` as explicit integers — scoring is deterministic and phrasing-independent
- `filter_grounded_threats` from `pipeline_utils` uses full-path matching (not substring matching)
- Retry appends a corrective message on JSON parse failure instead of replaying the same prompt
- Discard rate and per-component stats are logged for QA monitoring
- All shared logic lives in `pipeline_utils`; nothing is duplicated here

**Input:** `components.json`, `repo_surface.json`  
**Output:** `threats.json`

In [ ]:
!pip install -q anthropic

import os, sys
sys.path.insert(0, ".")

from pipeline_utils import (
    load_json, save_json, get_logger,
    filter_grounded_threats,
    parse_json_response, call_with_retry,
)

import json
from pathlib import Path
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
log    = get_logger("NB03")

## 3.1 — Load Inputs

In [ ]:
surface    = load_json("repo_surface.json",  schema_keys=["valid_paths", "run_id", "repo"])
comps_data = load_json("components.json",    schema_keys=["components", "data_flows"])

# Verify both artifacts are from the same pipeline run
if comps_data["metadata"]["run_id"] != surface["run_id"]:
    log.warning(
        "run_id mismatch: repo_surface.json=%s, components.json=%s. "
        "You may be mixing artifacts from different pipeline runs.",
        surface["run_id"], comps_data["metadata"]["run_id"]
    )

valid_paths: set[str] = set(surface["valid_paths"])   # full relative paths
RUN_ID = surface["run_id"]

STRIDE_LABELS = {
    "S": "Spoofing",
    "T": "Tampering",
    "R": "Repudiation",
    "I": "Information Disclosure",
    "D": "Denial of Service",
    "E": "Elevation of Privilege",
}

log.info("Components  : %d", len(comps_data["components"]))
log.info("Data flows  : %d", len(comps_data["data_flows"]))
log.info("Valid paths : %d", len(valid_paths))

## 3.2 — STRIDE System Prompt

The prompt explicitly requires `iae_factors` — nine integers (1–3) in each threat. This is the key architectural change from v1: scoring in NB04 reads structured fields, not prose keywords.

In [ ]:
STRIDE_SYSTEM = """
You are a senior application security engineer performing threat modelling.
For each STRIDE category, identify concrete, plausible threats for the given component.

Rules:
  1. Only report threats plausible given the component's actual code and function.
  2. Every threat MUST cite a specific file path and function name from the
     provided context in its 'evidence' field. Generic references like
     'the codebase' or 'the application' are NOT acceptable and will be discarded.
  3. If a STRIDE category has no plausible threat for this component, omit it.
  4. For each threat, assess the nine IAE factors and output them as explicit
     integers (1, 2, or 3) in the 'iae_factors' object.
  5. Respond ONLY with valid JSON — no markdown, no explanation.

IAE factor definitions (all scored 1=low, 2=medium, 3=high):
  Impact:
    data_sensitivity:    1=public, 2=user data, 3=credentials/PII/financial
    privilege_level:     1=none required, 2=authenticated user, 3=admin/arbitrary
    system_criticality:  1=non-critical, 2=core feature, 3=auth/payment/security
  Exploitability:
    access_vector:       1=local only, 2=internal network, 3=public internet
    exploit_input_control: 1=fully validated, 2=partial, 3=no validation/unsanitised
    attack_complexity:   1=complex multi-step, 2=moderate effort, 3=simple/trivial
  Exposure:
    endpoint_visibility: 1=internal only, 2=limited access, 3=public API
    auth_barrier:        1=strong MFA/JWT, 2=weak/basic auth, 3=no auth/bypassable
    data_flow_reach:     1=local component, 2=single service, 3=cross-system/DB

Response schema:
{
  "threats": [
    {
      "stride_category": "S|T|R|I|D|E",
      "title": "short threat name",
      "description": "concrete 1-2 sentence description of the threat",
      "evidence": "exact/relative/path/file.ext — function_name() — line or behaviour",
      "attack_vector": "step-by-step how an attacker exploits this",
      "affected_assets": ["list of data or services at risk"],
      "iae_factors": {
        "data_sensitivity": 2,
        "privilege_level": 1,
        "system_criticality": 3,
        "access_vector": 3,
        "exploit_input_control": 2,
        "attack_complexity": 2,
        "endpoint_visibility": 3,
        "auth_barrier": 2,
        "data_flow_reach": 2
      }
    }
  ]
}
"""

## 3.3 — Per-Component STRIDE Analysis

In [ ]:
def _build_component_prompt(component: dict, flows: list[dict]) -> str:
    flow_lines = "\n".join(
        f"  - {f['data']} via {f['protocol']} (authenticated={f['authenticated']})"
        for f in flows
    ) or "  (none)"
    return (
        f"Component   : {component['name']} (type={component['type']})\n"
        f"Description : {component['description']}\n"
        f"Data handled: {', '.join(component.get('data_handled', []))}\n"
        f"Source files: {', '.join(component.get('matched_paths', component.get('files', [])))}\n"
        f"Data flows:\n{flow_lines}\n\n"
        "Apply STRIDE. Every threat MUST include a real file path in evidence "
        "and explicit integer values in iae_factors. JSON only."
    )


def _call_claude(messages: list[dict]) -> list[dict]:
    """Single Claude API call — called via call_with_retry."""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=STRIDE_SYSTEM,
        messages=messages,
    )
    raw = parse_json_response(resp.content[0].text, context="NB03")
    return raw.get("threats", [])


# ── Main analysis loop ────────────────────────────────────────────────────────
all_flows      = comps_data["data_flows"]
stride_results = {}
total_raw      = 0
total_kept     = 0

for comp in comps_data["components"]:
    flows  = [
        f for f in all_flows
        if f["from"] == comp["id"] or f["to"] == comp["id"]
    ]
    prompt = _build_component_prompt(comp, flows)
    log.info("Analysing: %s (%d flows)", comp["name"], len(flows))

    try:
        raw_threats = call_with_retry(
            _call_claude,
            [{"role": "user", "content": prompt}],
            max_retries=3,
            retry_prompt_suffix=(
                "Your previous response was not valid JSON. "
                "Respond with ONLY the JSON object, no fences or explanation."
            ),
            logger=log,
        )
    except RuntimeError as exc:
        log.error("Skipping %s after retries: %s", comp["name"], exc)
        raw_threats = []

    total_raw += len(raw_threats)

    # Validate iae_factors presence — warn for any threat missing them
    for t in raw_threats:
        if not t.get("iae_factors"):
            log.warning(
                "Threat '%s' in component '%s' is missing iae_factors — "
                "scoring will use defaults",
                t.get("title", "?"), comp["name"]
            )

    grounded = filter_grounded_threats(
        raw_threats, valid_paths, label=comp["name"], logger=log
    )
    total_kept += len(grounded)

    stride_results[comp["id"]] = {
        "component_name": comp["name"],
        "component_type": comp["type"],
        "threats":        grounded,
        "discarded":      len(raw_threats) - len(grounded),
    }

discard_rate = (total_raw - total_kept) / max(total_raw, 1)
log.info("Total threats raw=%d  kept=%d  discarded=%d  (%.1f%%)",
         total_raw, total_kept, total_raw - total_kept, discard_rate * 100)

## 3.4 — Threat Summary by Component

In [ ]:
for comp_id, data in stride_results.items():
    threats = data["threats"]
    if not threats:
        continue
    print(f"\n── {data['component_name'].upper()} ({len(threats)} grounded threats) ──")
    by_cat: dict[str, list] = {}
    for t in threats:
        by_cat.setdefault(t.get("stride_category", "?"), []).append(t)
    for cat, cat_threats in sorted(by_cat.items()):
        label = STRIDE_LABELS.get(cat, "Unknown")
        print(f"  [{cat}] {label}")
        for t in cat_threats:
            loc = t.get("threat_location", "?")
            has_factors = bool(t.get("iae_factors"))
            print(f"        * {t['title']}")
            print(f"          Location: {loc}  | IAE factors: {'✓' if has_factors else '✗ MISSING'}")

## 3.5 — Save `threats.json`

In [ ]:
output = {
    "stride_results": stride_results,
    "metadata": {
        "run_id":              RUN_ID,
        "repo":                surface["repo"],
        "model":               "claude-sonnet-4-6",
        "total_threats_raw":   total_raw,
        "total_threats_kept":  total_kept,
        "total_discarded":     total_raw - total_kept,
        "discard_rate":        round(discard_rate, 3),
    },
}

save_json(output, "threats.json", run_id=RUN_ID)
print("Saved threats.json (grounded threats with iae_factors only)")